In [1]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
import scipy.sparse.linalg as sla
import scipy.sparse as sp
from ssh_chain_sq import PHOTONICChain

In [2]:
N = 10
gamma_zero = 0.03
fock_photon = 8
Omega_J = 1.0
Chi_J = 0.0
base_excitations = 1
k_states = (8 * N - 1)
Omega_C = 0.2
Chi_C = 0.6
# Chi_C_values = np.linspace(-0.75, 0.75, 20)
PBC = False
KERR = None          # e.g. KERR = 0.05 to turn the Kerr term on
STATISTICS = 'boson'  # 'boson' or 'fermion' ('fermion' ignores KERR)

gamma = gamma_zero
aux = 0.5 * Omega_C

omega_r = 1.065 * np.sqrt(1 + aux) - np.sqrt(1 - aux)
scale = np.exp(10 * np.abs(Chi_C))

chain = PHOTONICChain(N=N, Omega_C=Omega_C, Chi_C=Chi_C, omega_r=omega_r,
                        gamma=gamma, fock_photon=fock_photon,
                        Omega_J=Omega_J, Chi_J=Chi_J, kerr=KERR, PBC=PBC,
                        statistics=STATISTICS)

In [4]:
chain.single_particle_translation_op()

In [19]:
chain.transformed_translation_op.terms[2]

TensorTerm(tensor=array([[ 5.49349480e-02, -6.18410118e-03, -7.10465190e-04,
        -1.58380612e-03,  1.01467110e-03, -2.83545031e-04,
        -1.04785628e-03,  1.98856548e-04,  6.97436845e-04,
         1.44545581e-04, -2.53078157e-04,  6.50296007e-04,
         5.25505372e-04,  9.81273999e-04, -2.98402711e-04,
         9.54370324e-04, -5.63994259e-04,  6.70364249e-04,
         3.49476949e-03, -4.64070641e-04],
       [ 6.04596981e-03,  5.00373902e-02,  1.29237183e-02,
        -1.99086561e-03, -6.30932155e-03, -2.42724887e-03,
         4.49162227e-03, -2.11530784e-03, -2.56300297e-03,
        -8.69554416e-04,  9.19611638e-04, -2.12417394e-03,
        -2.23209099e-03, -3.57071913e-03,  2.55332931e-03,
        -4.62795761e-03, -2.08855038e-03, -8.33895740e-03,
         3.89824365e-03,  3.41660440e-03],
       [-6.69164837e-04, -1.24505462e-02,  4.74823357e-02,
         1.24256541e-02, -2.68001833e-03,  2.09347500e-03,
         2.76664437e-03, -9.19487991e-05, -1.84072824e-03,
        -2.

In [3]:
from sq_engine import SecondQuantizationEngine, QuantumOperator, TensorTerm, Superposition, CompositeEngine
N       = N
PBC = True
Omega_C = Omega_C
Chi_C   = Chi_C
gc1 = gc1 = 0.25 * Omega_C * (1 + Chi_C)
gc2 = gc2 = 0.25 * Omega_C * (1 - Chi_C)
Omega_J = Omega_J
Chi_J   = Chi_J
gj1 = gj1 =  Omega_J * (1 + Chi_J)
gj2 = gj2 = -Omega_J * (1 - Chi_J)
omega_r = omega_r
gamma   = gamma
kerr = None
fock_photon = fock_photon
statistics  = 'boson'
comp_engine = CompositeEngine()    
n = 2 * N
T = np.zeros((n, n))
for j in range(n - 1):
    val = gc1 if j % 2 == 0 else gc2
    T[j, j+1] = T[j+1, j] = val
if PBC and N != 1:
    T[0, n-1] = T[n-1, 0] = gc2

A  =  0.5 * (np.eye(n) + T)
B  = -0.5 * T

# ------------------------------------------------------------------
# Josephson Nambu matrices (identical to ssh_chain_single for bosons)
# ------------------------------------------------------------------
TJ = np.zeros((n, n))
for j in range(n - 1):
    val = gj1 if j % 2 == 0 else gj2
    TJ[j, j+1] = TJ[j+1, j] = val
if PBC and N != 1:
    TJ[0, n-1] = TJ[n-1, 0] = gj2

AJ =  0.5 * ((gj1 + gj2) * np.eye(n) - TJ)
BJ = -0.5 * TJ - 0.5 * (gj1 + gj2) * np.eye(n)

# ------------------------------------------------------------------
# Bogoliubov diagonalization via sq_engine
# diagonalize_quadratic expects H = A b†b + B_half b†b†  with
# Build a minimal operator
# carrying only the (1,0) and (1,1) blocks.
# ------------------------------------------------------------------

coeff_tol = 1e-9
zero_tol = 1e-12
engine = SecondQuantizationEngine(
    n_modes=n, statistics=statistics,
    coeff_tol=coeff_tol, zero_tol=zero_tol
)

def diagonalize(A, B, AJ, BJ):

    HC = QuantumOperator([
        TensorTerm(A,       (1, 0)),
        TensorTerm(B,       (1, 1)),
    ])

    if PBC:
        k_values = 2.0 * np.pi * np.fft.fftfreq(n)
        exponents = -1.0j * np.outer(np.arange(n), k_values)
        U_k = (1 / np.sqrt(n)) * np.exp(exponents)   # now n x n
        V_k = np.zeros_like(U_k)
        H_C_for_diag = engine.change_basis(HC, U_k, V_k).simplify(tol = 1e-9)
    else:
        H_C_for_diag = HC
    return H_C_for_diag

In [4]:
hc = diagonalize(A, B, AJ, BJ)

In [5]:
hc.terms[1]

TensorTerm(tensor=array([[-5.00000000e-02+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j],
       [ 0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000e+00+0.00000000e+00j,  0.00000000e+00+0.00000000e+00j,
         0.00000000